# Movie ABT Enrichment - Agent-Based Version with LangGraph

## Why Use an Agent Approach?

This version demonstrates **when agents add value** to data enrichment:

### Agent Advantages:
1. **Intelligent Source Selection**: Agent decides which data source to use based on availability
2. **Adaptive Extraction**: If one method fails, agent tries alternatives
3. **Quality Validation**: Agent verifies data completeness and retries if needed
4. **Error Recovery**: Automatic retry with different strategies
5. **Reasoning**: Agent can explain why it chose certain sources or methods

### Comparison:
- **Simple Version** (previous): Fast, deterministic, good for clean data
- **Agent Version** (this): Adaptive, resilient, good for messy/incomplete data

Use this when:
- Data sources are unreliable
- You need quality validation
- Multiple extraction strategies might be needed
- You want transparency in decision-making

In [ ]:
# Install packages
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai beautifulsoup4 requests --quiet
print("✓ Packages installed")

In [1]:
import os
import pandas as pd
import requests
import json
import time
from pathlib import Path
from typing import Dict, Optional, Literal

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

print("✓ Imports successful")

✓ Imports successful


In [2]:
load_dotenv()
load_dotenv('../.env')

# Use GPT-4o for better reasoning
llm = ChatOpenAI(model="gpt-4o", temperature=0)
print("✓ LLM initialized (GPT-4o for agent reasoning)")

✓ LLM initialized (GPT-4o for agent reasoning)


## Configuration

In [7]:
INPUT_FILE = "../data/outputs/analytics_base_table_with_genres_list.csv"
OUTPUT_FILE = "analytics_base_table_ENRICHED_AGENT.xlsx"
MAX_MOVIES = 10  # Start small to see agent in action
SLEEP_BETWEEN_REQUESTS = 2.0

print(f"✓ Configuration: Processing {MAX_MOVIES if MAX_MOVIES else 'all'} movies")

✓ Configuration: Processing 10 movies


## Define Agent Tools

In [8]:
@tool
def fetch_tmdb_data(tmdb_id: str) -> str:
    """
    Fetch comprehensive movie data from TMDB (The Movie Database).
    
    Use this tool when you need:
    - Financial data (budget, revenue)
    - Detailed production info (status, tagline)
    - International language data
    - Popularity metrics
    
    Args:
        tmdb_id: The TMDB movie ID (numeric string)
    
    Returns:
        JSON string with movie data or error message
    """
    try:
        url = f"https://www.themoviedb.org/movie/{tmdb_id}"
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # Extract with LLM
        extraction_prompt = f"""
Extract movie data from this TMDB page as JSON.

Fields: title, original_title, overview, tagline, status, release_date,
runtime, budget, revenue, popularity, genres, original_language, spoken_languages

HTML: {response.text[:8000]}

Return ONLY valid JSON.
"""
        
        result = llm.invoke([HumanMessage(content=extraction_prompt)])
        json_str = result.content.strip()
        
        if json_str.startswith('```'):
            json_str = json_str.split('```')[1]
            if json_str.startswith('json'):
                json_str = json_str[4:]
            json_str = json_str.strip()
        
        # Validate JSON
        data = json.loads(json_str)
        return json.dumps({"success": True, "source": "TMDB", "data": data})
        
    except Exception as e:
        return json.dumps({"success": False, "source": "TMDB", "error": str(e)})


@tool
def fetch_imdb_data(imdb_id: str) -> str:
    """
    Fetch movie data from IMDB (Internet Movie Database).
    
    Use this tool when you need:
    - User ratings and reviews
    - Cast information
    - Poster images
    - Keywords and tags
    
    Args:
        imdb_id: The IMDB movie ID (format: 'tt0123456' or just '0123456')
    
    Returns:
        JSON string with movie data or error message
    """
    try:
        # Format ID
        if not imdb_id.startswith('tt'):
            imdb_id = f"tt{imdb_id.zfill(7)}"
        
        url = f"https://www.imdb.com/title/{imdb_id}/"
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        extraction_prompt = f"""
Extract movie data from this IMDB page as JSON.

Fields: id, title, genres, rating, cast, overview, runtime,
release_year, poster_url, keywords

HTML: {response.text[:8000]}

Return ONLY valid JSON.
"""
        
        result = llm.invoke([HumanMessage(content=extraction_prompt)])
        json_str = result.content.strip()
        
        if json_str.startswith('```'):
            json_str = json_str.split('```')[1]
            if json_str.startswith('json'):
                json_str = json_str[4:]
            json_str = json_str.strip()
        
        data = json.loads(json_str)
        return json.dumps({"success": True, "source": "IMDB", "data": data})
        
    except Exception as e:
        return json.dumps({"success": False, "source": "IMDB", "error": str(e)})


@tool
def validate_movie_data(data: str) -> str:
    """
    Validate that movie data is complete and high-quality.
    
    Use this tool to check if fetched data has:
    - Required fields (title, overview)
    - Reasonable values (runtime > 0, rating in valid range)
    - No obvious errors or missing data
    
    Args:
        data: JSON string of movie data to validate
    
    Returns:
        JSON with validation results and recommendations
    """
    try:
        parsed = json.loads(data)
        
        issues = []
        score = 100
        
        # Check required fields
        if not parsed.get('title'):
            issues.append("Missing title")
            score -= 30
        
        if not parsed.get('overview'):
            issues.append("Missing overview")
            score -= 20
        
        # Check data quality
        if 'runtime' in parsed and (not parsed['runtime'] or parsed['runtime'] <= 0):
            issues.append("Invalid runtime")
            score -= 10
        
        if 'rating' in parsed:
            rating = parsed['rating']
            if rating and (rating < 0 or rating > 10):
                issues.append("Invalid rating")
                score -= 10
        
        result = {
            "valid": score >= 50,
            "quality_score": score,
            "issues": issues,
            "recommendation": "Accept" if score >= 70 else "Retry from alternate source" if score >= 40 else "Data too incomplete"
        }
        
        return json.dumps(result)
        
    except Exception as e:
        return json.dumps({"valid": False, "error": str(e)})


print("✓ Agent tools defined:")
print("  • fetch_tmdb_data")
print("  • fetch_imdb_data")
print("  • validate_movie_data")

✓ Agent tools defined:
  • fetch_tmdb_data
  • fetch_imdb_data
  • validate_movie_data


## Create ReAct Agent

In [9]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage

# Define tools list
tools = [fetch_tmdb_data, fetch_imdb_data, validate_movie_data]

# System prompt (we'll include this in EVERY agent call)
system_prompt = """
You are a movie data enrichment agent. Your job is to fetch comprehensive, 
high-quality movie data from TMDB and IMDB.

DECISION-MAKING STRATEGY:
1. Start with BOTH sources (TMDB and IMDB) in parallel if possible
2. Validate the quality of data from each source
3. If one source fails, try the other
4. If both succeed, combine their data
5. If data quality is low, retry with alternate extraction methods

QUALITY STANDARDS:
- Must have: title, overview
- Should have: runtime, genres, release date/year
- Nice to have: cast, budget, ratings

RETURN FORMAT:
Return a JSON object with:
- tmdb_data: {all TMDB fields}
- imdb_data: {all IMDB fields}
- quality_assessment: {quality score and notes}
- sources_used: ["TMDB", "IMDB"]
"""

# Create the agent (CORRECTED - no state_modifier parameter)
agent = create_react_agent(llm, tools)

print("✓ ReAct agent created with reasoning capabilities")

✓ ReAct agent created with reasoning capabilities


/var/folders/7m/mfqtd68j7pl0t758kp_14m3h0000gn/T/ipykernel_72785/2354567290.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


## Load and Process Data

In [10]:
# Read ABT file
input_path = Path(INPUT_FILE)
if not input_path.exists():
    input_path = Path("../data") / INPUT_FILE

df = pd.read_csv(input_path) if input_path.suffix == '.csv' else pd.read_excel(input_path)

if MAX_MOVIES:
    df = df.head(MAX_MOVIES)

print(f"✓ Loaded {len(df)} movies to process")
df.head(3)

✓ Loaded 10 movies to process


,Unnamed: 0,movieId,title,genres,avg rating,no of viewer rating,viewers tags,imdbId,tmdbId,Genres_List
0,0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.9,215.0,pixar|pixar|fun,114709,862.0,"['Adventure', 'Animation', 'Children', 'Comedy..."
1,1,2,Jumanji (1995),Adventure|Children|Fantasy,3.4,110.0,fantasy|magic board game|Robin Williams|game,113497,8844.0,"['Adventure', 'Children', 'Fantasy']"
2,2,3,Grumpier Old Men (1995),Comedy|Romance,3.3,52.0,moldy|old,113228,15602.0,"['Comedy', 'Romance']"


In [11]:
# Initialize new columns
tmdb_columns = [
    'tmdb_title', 'tmdb_original_title', 'tmdb_overview', 'tmdb_tagline',
    'tmdb_status', 'tmdb_release_date', 'tmdb_runtime', 'tmdb_budget',
    'tmdb_revenue', 'tmdb_popularity', 'tmdb_genres',
    'tmdb_original_language', 'tmdb_spoken_languages'
]

imdb_columns = [
    'imdb_id', 'imdb_title', 'imdb_genres', 'imdb_rating',
    'imdb_cast', 'imdb_overview', 'imdb_runtime',
    'imdb_release_year', 'imdb_poster_url', 'imdb_keywords'
]

agent_columns = ['agent_quality_score', 'agent_sources_used', 'agent_notes']

for col in tmdb_columns + imdb_columns + agent_columns:
    df[col] = None

print(f"✓ Added {len(tmdb_columns + imdb_columns + agent_columns)} new columns")

✓ Added 26 new columns


In [12]:
# Process with agent
print("="*70)
print("AGENT-BASED ENRICHMENT")
print("="*70)
print("Watch the agent reason through each movie!\n")

for idx, row in df.iterrows():
    movie_title = row.get('title', 'Unknown')
    tmdb_id = row.get('tmdbId')
    imdb_id = row.get('imdbId')
    
    print(f"\n{'='*70}")
    print(f"[{idx+1}/{len(df)}] {movie_title}")
    print(f"TMDB ID: {tmdb_id} | IMDB ID: {imdb_id}")
    print("="*70)
    
    # Format IMDB ID
    imdb_id_str = str(int(imdb_id)) if pd.notna(imdb_id) else None
    if imdb_id_str and not imdb_id_str.startswith('tt'):
        imdb_id_str = f"tt{imdb_id_str.zfill(7)}"
    
    # Create agent query
    query = f"""
Enrich data for the movie: {movie_title}
TMDB ID: {tmdb_id if pd.notna(tmdb_id) else 'not available'}
IMDB ID: {imdb_id_str if imdb_id_str else 'not available'}

Fetch data from both sources, validate quality, and return comprehensive results.
"""
    
    try:
        # Invoke agent
        result = agent.invoke({"messages": [HumanMessage(content=query)]})
        
        # Extract response
        final_message = result["messages"][-1].content
        
        print("\n🤖 Agent Response:")
        print("-"*70)
        print(final_message[:500])  # Show first 500 chars
        if len(final_message) > 500:
            print("...")
        
        # Try to extract JSON from response
        try:
            if '```json' in final_message:
                json_str = final_message.split('```json')[1].split('```')[0].strip()
            elif '```' in final_message:
                json_str = final_message.split('```')[1].split('```')[0].strip()
            else:
                # Try to find JSON in the text
                json_start = final_message.find('{')
                json_end = final_message.rfind('}') + 1
                if json_start >= 0 and json_end > json_start:
                    json_str = final_message[json_start:json_end]
                else:
                    raise ValueError("No JSON found in response")
            
            enriched_data = json.loads(json_str)
            
            # Extract TMDB data
            if 'tmdb_data' in enriched_data and enriched_data['tmdb_data']:
                for key, value in enriched_data['tmdb_data'].items():
                    col_name = f"tmdb_{key}"
                    if col_name in df.columns:
                        df.at[idx, col_name] = value
            
            # Extract IMDB data
            if 'imdb_data' in enriched_data and enriched_data['imdb_data']:
                for key, value in enriched_data['imdb_data'].items():
                    col_name = f"imdb_{key}"
                    if col_name in df.columns:
                        df.at[idx, col_name] = value
            
            # Store agent metadata
            if 'quality_assessment' in enriched_data:
                df.at[idx, 'agent_quality_score'] = enriched_data['quality_assessment'].get('quality_score')
                df.at[idx, 'agent_notes'] = enriched_data['quality_assessment'].get('notes')
            
            if 'sources_used' in enriched_data:
                df.at[idx, 'agent_sources_used'] = ', '.join(enriched_data['sources_used'])
            
            print("\n✓ Data enriched successfully")
            
        except Exception as e:
            print(f"\n⚠ Could not parse agent response as JSON: {e}")
            df.at[idx, 'agent_notes'] = f"Parse error: {str(e)}"
    
    except Exception as e:
        print(f"\n❌ Agent failed: {e}")
        df.at[idx, 'agent_notes'] = f"Agent error: {str(e)}"
    
    time.sleep(SLEEP_BETWEEN_REQUESTS)

print("\n" + "="*70)
print("AGENT ENRICHMENT COMPLETE")
print("="*70)

AGENT-BASED ENRICHMENT
Watch the agent reason through each movie!


[1/10] Toy Story (1995)
TMDB ID: 862.0 | IMDB ID: 114709

🤖 Agent Response:
----------------------------------------------------------------------
Here is the enriched data for the movie "Toy Story" (1995) along with validation results:

### TMDB Data
- **Title**: Toy Story
- **Original Title**: Toy Story
- **Overview**: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.
- **Validation**: 
 
...

⚠ Could not parse agent response as JSON: No JSON found in response

[2/10] Jumanji (1995)
TMDB ID: 8844.0 | IMDB ID: 113497

🤖 Agent Response:
----------------------------------------------------------------------
Here is the enriched data for the movie "Jumanji" (1995) along w

In [13]:
# Save enriched data
output_path = input_path.parent / OUTPUT_FILE
all_new_cols = tmdb_columns + imdb_columns + agent_columns

df.to_excel(output_path, index=False, engine='openpyxl')

print(f"✓ Saved to: {output_path}")
print(f"✓ Total columns: {len(df.columns)}")
print(f"  - Original: {len(df.columns) - len(all_new_cols)}")
print(f"  - TMDB: {len(tmdb_columns)}")
print(f"  - IMDB: {len(imdb_columns)}")
print(f"  - Agent metadata: {len(agent_columns)}")

✓ Saved to: ../data/outputs/analytics_base_table_ENRICHED_AGENT.xlsx
✓ Total columns: 36
  - Original: 10
  - TMDB: 13
  - IMDB: 10
  - Agent metadata: 3


## Agent vs Simple Comparison

### Simple Version (4_Movie_ABT_Enrichment):
- ✅ **Faster**: Direct HTTP → LLM extraction
- ✅ **Predictable**: Same process for every movie
- ✅ **Cheaper**: Uses gpt-4o-mini
- ❌ No reasoning or adaptation
- ❌ No quality validation

### Agent Version (this notebook):
- ✅ **Adaptive**: Tries multiple strategies
- ✅ **Validated**: Checks data quality
- ✅ **Resilient**: Recovers from errors
- ✅ **Transparent**: Explains decisions
- ❌ Slower (reasoning overhead)
- ❌ More expensive (gpt-4o + multiple calls)

### When to Use Each:
- **Simple**: Clean IDs, reliable sources, large batch (1000s movies)
- **Agent**: Messy data, need quality validation, willing to trade speed for reliability